In [30]:
import pandas as pd
import geopandas as gpd
from glob import glob

In [31]:
files = glob("Datos/antenas/*.geojson")
files

['Datos/antenas\\1.geojson',
 'Datos/antenas\\10.geojson',
 'Datos/antenas\\2.geojson',
 'Datos/antenas\\3.geojson',
 'Datos/antenas\\4.geojson',
 'Datos/antenas\\5.geojson',
 'Datos/antenas\\6.geojson',
 'Datos/antenas\\7.geojson',
 'Datos/antenas\\8.geojson',
 'Datos/antenas\\9.geojson']

In [32]:
antenas = gpd.GeoDataFrame()
for file in files:
    df = gpd.read_file(file)
    antenas = pd.concat([antenas, df], ignore_index=True)
print(antenas.shape)
antenas.drop_duplicates(inplace=True)
antenas.reset_index(drop=True, inplace=True)
antenas = antenas[antenas['geometry'].geom_type == 'Point']
antenas = antenas[antenas['geometry'].notnull()]
print(antenas.shape)
antenas.head(1)

(6301, 64)
(6205, 64)


,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.OBJECTID,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.OBJECTID_1,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.ALIAS,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.SOPO_ID_EMPRESA,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.TISO_DESCRIPCION,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.SOPO_ALTURA,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.DTE_DIRECCION,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.COMUNA,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.REGION,ARCGIS.LDT_ELEMENTOS_AUT_PT_1.DTE_LAT_SUR_GRA,...,ARCGIS.LDT_SOPORTES_AUT_PT_1.TIEM_GLOSA,ARCGIS.LDT_SOPORTES_AUT_PT_1.LATITUD_CALCULADA,ARCGIS.LDT_SOPORTES_AUT_PT_1.LONGITUD_CALCULADA,ARCGIS.LDT_SOPORTES_AUT_PT_1.INAMI_GLOSA,ARCGIS.LDT_SOPORTES_AUT_PT_1.TIAC_GLOSA,ARCGIS.LDT_SOPORTES_AUT_PT_1.LOCALIDAD,ARCGIS.LDT_SOPORTES_AUT_PT_1.TIESTN_GLOSA,ARCGIS.LDT_SOPORTES_AUT_PT_1.DIRA_GLOSA,ARCGIS.LDT_SOPORTES_AUT_PT_1.ID_PROCESO,geometry
0,10,10,ENTEL PCS,EPCS0083,Sin Info,18.0,CAUPOLICAN Nº 997,Arica,Región de Arica y Parinacota,18,...,Urbano,-18.484167,-70.305833,Sin Info,Sin Info,None,Celda,DIRECCIONAL,None,POINT (-70.30583 -18.48417)


In [33]:
antenas.rename(columns={
    # Información de frecuencia y banda
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.FRECUENCIA': 'frecuencia_operacion_hz',
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.BANDA': 'banda_frecuencia',
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.ANCHOBANDA': 'ancho_banda_hz',
    
    # Tecnología y características de transmisión
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.TECNOLOGIA': 'tecnologia_transmision',
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.TIPOACCESO': 'tipo_acceso_red',

    # Posición geográfica (importante para análisis de interferencia)
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.LATITUD_CALCULADA': 'latitud',
    'ARCGIS.LDT_ELEMENTOS_AUT_PT_1.LONGITUD_CALCULADA': 'longitud'
}, inplace=True)

In [34]:
antenas = antenas[['frecuencia_operacion_hz', 'banda_frecuencia',
                   'ancho_banda_hz', 'tecnologia_transmision',
                   'tipo_acceso_red', 'latitud', 'longitud']]
antenas

,frecuencia_operacion_hz,banda_frecuencia,ancho_banda_hz,tecnologia_transmision,tipo_acceso_red,latitud,longitud
0,1900.0,UHF,30000,GSM(2G),TDMA,-18.484167,-70.305833
1,1900.0,UHF,30000,GSM(2G),TDMA,-18.454167,-70.289167
2,1900.0,UHF,30000,GSM(2G),TDMA,-18.470833,-70.294167
3,1900.0,UHF,30000,LTE(4G)/GSM(2G),FDD,-18.470833,-70.294167
4,850.0,UHF,25000,GSM(2G),TDMA,-19.813056,-69.962222
...,...,...,...,...,...,...,...
6200,3500.0,SHF,50000,5G(5G),None,-31.119944,-71.158417
6201,700.0,UHF,10000,LTE(4G),FDD,-30.630000,-71.191361
6202,700.0,UHF,10000,LTE(4G),FDD,-30.630000,-71.191361
6203,700.0,UHF,10000,LTE(4G),FDD,-31.321389,-71.324167


In [35]:
antenas['geometry'] = antenas.apply(lambda row: gpd.points_from_xy([row['longitud']], [row['latitud']])[0], axis=1)
antenas = gpd.GeoDataFrame(antenas, geometry='geometry', crs='EPSG:4326')
antenas = antenas.set_geometry('geometry')
antenas.drop(columns=['latitud', 'longitud'], inplace=True)
antenas = antenas[antenas['geometry'].notnull()]
antenas.reset_index(drop=True, inplace=True)
print(antenas.shape)
antenas.head(1)

(6205, 6)


,frecuencia_operacion_hz,banda_frecuencia,ancho_banda_hz,tecnologia_transmision,tipo_acceso_red,geometry
0,1900.0,UHF,30000,GSM(2G),TDMA,POINT (-70.30583 -18.48417)


In [36]:
antenas.to_file("Datos/antenas/antenas.shp", driver='ESRI Shapefile')

C:\Users\Tomy\AppData\Local\Temp\ipykernel_23936\2618657987.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  antenas.to_file("Datos/antenas/antenas.shp", driver='ESRI Shapefile')
c:\Users\Tomy\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'frecuencia_operacion_hz' to 'frecuencia'
  ogr_write(
c:\Users\Tomy\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'banda_frecuencia' to 'banda_frec'
  ogr_write(
c:\Users\Tomy\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ancho_banda_hz' to 'ancho_band'
  ogr_write(
c:\Users\Tomy\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'tecnologia_transmision' to 'tecnologia'
  ogr_wr